# 07 — Cohort A — count the catalogue

| | |
|---|---|
| **runs at** | cohort A |
| **reads** | `cohort-A.csv`, `federated_pvclust_catalogue.csv` |
| **writes** | `cohort-A_counts.csv` |

The second pass, and the one that makes federated *p*-values possible.

This cohort re-runs its bootstrap, but instead of tallying the clusters of its
own tree, it tallies **the aggregator's clusters**. Every cohort is now
answering the same question about the same clusters, so the tallies can be
added.

A cohort can and will count a cluster its own tree never produced. That is the
point: it is how a cohort contributes evidence about something it could not
have found alone.

Only the counts leave — one integer per cluster per resampling scale.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
# The SLE data is not in the repository. Download it from Zenodo
# (doi:10.5281/zenodo.20342569), unpack it, and point SLE at the folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 80        # 80 objects keeps p < n in BOTH cohorts
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
else:
    # A stand-in of the same shape, so every cell runs without the download:
    # two batches of unequal size, a case/control split, and a few proteins
    # measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    FEATURE_MAP = 'feature_metadata.txt'
print(f"{'SLE data' if REAL else 'stand-in'}: n_boot={NBOOT}, top_variable={TOPVAR}")

In [ ]:
run(['pvclust-py', 'count-edges', '--project', 'cohort-A',
     '--catalogue', 'federated_pvclust_catalogue.csv',
     '--matrix', 'cohort-A.csv', '--log2',
     '--metadata', 'cohort-A_meta.csv',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--shared-features', 'shared_features.csv',
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol',
     '--dist', 'minkowski', '--linkage', 'ward.D2',
     '--n-boot', str(NBOOT)])

## What is in the file that leaves

One row per cluster per scale. The `n` column is **not decoration**: each
cohort's resampling scales are relative to its own row count, so the aggregator
needs `n` to put them on a common footing before summing. Without it the pooled
curve is fitted against the wrong abscissa and federated AU collapses to zero.

In [ ]:
c = pd.read_csv('cohort-A_counts.csv')
print(c.head(4).to_string(index=False))
print()
print(f'{c.edge_id.nunique()} clusters x {c.r.nunique()} scales = {len(c)} rows')
print(f'this cohort recovered {c["count"].sum()} of {c.nboot.sum()} draws overall')

---
**Next:** `09`. The aggregator sums both cohorts' counts and fits the curve.